# Masking delle feature FC in funzione della lesione

**Obiettivo**: per ciascun soggetto, azzerare le righe/colonne dei nodi (parcel) della matrice di connettività funzionale (FC) che si sovrappongono in modo sostanziale alla lesione — stessa logica del filone Siegel/Griffis (Corbetta lab, WashU) gia' usato su questa stessa coorte.

**Riferimenti**:
- Siegel et al. 2016, *PNAS* — connessioni di parcel lesionati rimosse (univariate) / azzerate (multivariate).
- Griffis et al. 2019, *Neuron* (gia' in `assets/papers/`) — versione piu' precisa: vertici lesionati mascherati, parcel con troppo pochi vertici sani rimasti escluso interamente, soglia comune in letteratura ~50% di overlap (Lesion Quantification Toolkit, Griffis et al. 2021).
- **XCP-D** (`xcp_d/interfaces/connectivity.py`, classe `NiftiParcellate`) — lo stesso tool che ha generato queste matrici FC implementa gia' un meccanismo identico (`min_coverage`, default 0.5) per la copertura BOLD; qui riusiamo lo stesso schema con la maschera di lesione al posto della maschera di copertura BOLD.

**Decisione presa con l'utente**: niente funzione di parcellizzazione scritta a mano — si riusa `nilearn.maskers.NiftiLabelsMasker` (stessa libreria gia' usata altrove nel repo), con lo stesso schema a doppio masker (con/senza maschera) di XCP-D.

In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import glob

import nibabel as nib
import numpy as np
import pandas as pd
from nilearn.image import resample_to_img
from nilearn.maskers import NiftiLabelsMasker

SUBJECT = "sub-STUNIPD0003"
ATLAS_COMBO = "atlas-Yan200TianS2Buckner7N"
MIN_COVERAGE = 0.5  # soglia standard di campo (LQT, XCP-D default)

DATA_ROOT = "../data/clinical_connectome/derivatives/UNIPD/WashU"
ATLAS_ROOT = f"../assets/atlases/fmriprep/{ATLAS_COMBO}"

## 1. Atlante: BIDS-Derivatives gia' pronto sul server

`Atlases/fmriprep/atlas-Yan200TianS2Buckner7N/` (server EBRAIN) contiene gia': `*_dseg.tsv` (lookup ufficiale index->nome nodo) + `*_res-2_dseg.nii.gz` (volume 2mm, stesso spazio `MNI152NLin6Asym` della lesion mask WashU — nessun resampling di spazio necessario, solo eventuale allineamento di griglia).

In [3]:
atlas_img = nib.squeeze_image(
    nib.load(f"{ATLAS_ROOT}/{ATLAS_COMBO}_space-MNI152NLin6Asym_res-2_dseg.nii.gz")
)
label_table = pd.read_csv(f"{ATLAS_ROOT}/{ATLAS_COMBO}_dseg.tsv", sep="\t")
label_ids = label_table["index"].tolist()
id_to_name = dict(zip(label_table["index"], label_table["label"]))

print(f"Atlante caricato: {len(label_ids)} nodi")
label_table.head(3)

Atlante caricato: 239 nodi


,index,label
0,1,7Networks_LH_Default_IPL_1
1,2,7Networks_LH_Default_IPL_2
2,3,7Networks_LH_Default_IPL_3


## 2. Lesion mask: resample obbligatorio, mai per indice

La lesion mask WashU e l'atlante hanno la **stessa shape** (91, 109, 91) ma affini diverse (segno dell'asse X opposto — orientamento radiologico vs neurologico). Stessa shape non implica stesso voxel-to-world mapping: senza resample esplicito via affine si rischia un flip L/R silenzioso.

In [4]:
lesion_path = (
    f"{DATA_ROOT}/manual_masks/{SUBJECT}/anat/"
    f"{SUBJECT}_space-MNI152NLin6Asym_label-lesion_mask.nii.gz"
)
lesion_img = nib.load(lesion_path)

print("Lesion mask affine (WashU):\n", lesion_img.affine)
print("Atlas affine (server, res-2):\n", atlas_img.affine)
print("-> stessa shape, segno asse X diverso: resample via affine obbligatorio.")

lesion_resampled = resample_to_img(
    lesion_img, atlas_img, interpolation="nearest", force_resample=True, copy_header=True
)
lesion_data = (np.asarray(lesion_resampled.get_fdata()) > 0.5).astype(np.int32)
print(f"\nVoxel lesionati ({SUBJECT}, griglia atlante 2mm): {int(lesion_data.sum())}")

Lesion mask affine (WashU):
 [[  -2.    0.    0.   90.]
 [   0.    2.    0. -126.]
 [   0.    0.    2.  -72.]
 [   0.    0.    0.    1.]]
Atlas affine (server, res-2):
 [[   2.    0.    0.  -90.]
 [   0.    2.    0. -126.]
 [   0.    0.    2.  -72.]
 [   0.    0.    0.    1.]]
-> stessa shape, segno asse X diverso: resample via affine obbligatorio.

Voxel lesionati (sub-STUNIPD0003, griglia atlante 2mm): 7042


## 3. Copertura per parcel (schema XCP-D: doppio `NiftiLabelsMasker`)

Un masker senza maschera conta tutti i voxel del parcel; un secondo masker con `mask_img=healthy_img` (l'inverso della lesione) conta solo i voxel sani. Il rapporto e' la frazione di copertura sana per parcel — esattamente l'inverso del `fraction_lesioned` gia' usato in `src/features/lesion.py`, ma calcolato con la libreria invece che con un loop scritto a mano.

**Due bug reali trovati e corretti durante la validazione, non ovvi dalla documentazione nilearn:**

1. **Overflow silenzioso in `uint8`**: se l'immagine "contatore" (tutti 1) usata con `strategy="sum"` e' in `uint8`, la somma va in overflow a 256 (es. 695 voxel -> 183, cioe' `695 mod 256`). Fix: usare `int32`.
2. **Nodo completamente lesionato sparisce, non diventa 0**: se un parcel non ha *nessun* voxel sano rimasto, `NiftiLabelsMasker(mask_img=...)` lo **rimuove** dall'output invece di restituire 0 — mismatch di shape tra masker mascherato e non mascherato se non gestito esplicitamente. Fix: indicizzare per *nome del label*, non per posizione, e trattare un'assenza come coverage=0.0.

In [5]:
def compute_parcel_coverage(atlas_img, label_ids, healthy_img):
    """Frazione di voxel sani per parcel, indicizzata da label_ids (in quell'ordine).

    Un parcel con zero voxel sani rimasti viene rimosso da NiftiLabelsMasker quando si
    passa mask_img (non restituito come 0) - gestito qui esplicitamente come coverage
    0.0, mai lasciato disallineare silenziosamente il masker mascherato da quello no.
    """
    # int32, non uint8: NiftiLabelsMasker(strategy="sum") va in overflow silenzioso a 256
    # con un contatore troppo stretto (bug verificato su questi stessi dati).
    ones_img = nib.Nifti1Image(np.ones(atlas_img.shape, dtype=np.int32), atlas_img.affine, atlas_img.header)

    masker_total = NiftiLabelsMasker(labels_img=atlas_img, background_label=0, strategy="sum", standardize=False)
    n_total = np.squeeze(masker_total.fit_transform(ones_img))
    total_by_label = dict(zip(masker_total.labels_[1:], n_total))  # [1:]: labels_[0] e' il placeholder "Background"

    masker_healthy = NiftiLabelsMasker(
        labels_img=atlas_img, mask_img=healthy_img, background_label=0, strategy="sum", standardize=False
    )
    n_healthy = np.squeeze(masker_healthy.fit_transform(ones_img))
    healthy_by_label = dict(zip(masker_healthy.labels_[1:], np.atleast_1d(n_healthy)))

    return np.array([healthy_by_label.get(lbl, 0.0) / total_by_label[lbl] for lbl in label_ids])


healthy_img = nib.Nifti1Image((1 - lesion_data), atlas_img.affine, atlas_img.header)
parcel_coverage = compute_parcel_coverage(atlas_img, label_ids, healthy_img)
compromised = parcel_coverage < MIN_COVERAGE
node_names = np.array([id_to_name[i] for i in label_ids])

print(f"Nodi totali: {len(node_names)} | compromessi (coverage sana < {MIN_COVERAGE}): {int(compromised.sum())}\n")
for name, cov in sorted(zip(node_names[compromised], parcel_coverage[compromised]), key=lambda x: x[1]):
    print(f"  {name}: coverage sana={cov:.3f} (overlap lesione={1 - cov:.1%})")

Nodi totali: 239 | compromessi (coverage sana < 0.5): 8

  Tian_pPUT-lh: coverage sana=0.000 (overlap lesione=100.0%)
  Tian_aGP-lh: coverage sana=0.019 (overlap lesione=98.1%)
  Tian_pGP-lh: coverage sana=0.067 (overlap lesione=93.3%)
  Tian_pCAU-lh: coverage sana=0.128 (overlap lesione=87.2%)
  Tian_aPUT-lh: coverage sana=0.295 (overlap lesione=70.5%)
  Tian_aCAU-lh: coverage sana=0.463 (overlap lesione=53.7%)
  7Networks_LH_SomMot_Ins_2: coverage sana=0.466 (overlap lesione=53.4%)
  7Networks_LH_SalVentAttn_Ins_1: coverage sana=0.499 (overlap lesione=50.1%)


Risultato coerente con l'anatomia: `sub-STUNIPD0003` ha una lesione dei gangli della base sinistri (putamen, pallido, caudato — nodi `Tian_*-lh`), con overlap che degrada dal 100% (putamen posteriore) al 50% (insula), esattamente il pattern atteso per uno stroke sottocorticale.

## 4. Applicazione alla matrice FC reale

Azzeramento (stile Siegel et al.) delle righe/colonne dei nodi compromessi — allineamento **per nome nodo**, mai per posizione, verificato esplicitamente contro l'ordine dell'atlante.

In [6]:
fc_path = glob.glob(f"{DATA_ROOT}/features/{SUBJECT}/func/*{ATLAS_COMBO}*.csv")[0]
fc = pd.read_csv(fc_path, sep="\t", index_col=0)

assert list(fc.index) == list(node_names), "ordine nodi FC non combacia con l'atlante - mai allineare per posizione"
print(f"Matrice FC caricata: {fc.shape}, allineamento nomi nodo verificato.\n")

fc_masked = fc.copy()
compromised_names = node_names[compromised]
fc_masked.loc[compromised_names, :] = 0.0
fc_masked.loc[:, compromised_names] = 0.0

example_node = compromised_names[0]
print(f"Esempio - riga del nodo compromesso '{example_node}', originale vs mascherata:")
print("originale: ", fc.loc[example_node].values[:6])
print("mascherata:", fc_masked.loc[example_node].values[:6])

print(f"\nValori non-zero nella matrice: originale={int((fc.values != 0).sum())}, mascherata={int((fc_masked.values != 0).sum())}")

Matrice FC caricata: (239, 239), allineamento nomi nodo verificato.

Esempio - riga del nodo compromesso '7Networks_LH_SalVentAttn_Ins_1', originale vs mascherata:
originale:  [-0.11554876  0.0087889  -0.10546405 -0.1090772  -0.16777018 -0.16787197]
mascherata: [0. 0. 0. 0. 0. 0.]

Valori non-zero nella matrice: originale=57121, mascherata=53361


## Prossimo passo

Questo prototipo va validato su piu' soggetti/combinazioni di atlante (i 12 `Yan{100..400}TianS{1..3}Buckner7N`), poi migrato in `src/features/functional.py` seguendo lo stesso pattern di migrazione gia' usato per `notebooks/lesion_analysis.ipynb` -> `src/features/lesion.py`: stesso algoritmo, ristrutturato in funzioni tipizzate e testate secondo `code_standards.md`.